In [1]:
import pandas as pd
import os
from shapely.validation import make_valid 
from shapely.geometry import Point, MultiPolygon  
import pathlib
from tqdm import tqdm
import geopandas as gpd 

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [2]:
stations = pd.read_parquet("GEBCO/merged_all_gebco.parquet")

cols = [
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "datasetKey",
    "datasetTitle",
    "sourceDatabase",
    "year",
    "gebcoDepth",
    "startDayOfYear",
    "gear", 
]
all_stations = stations[cols].drop_duplicates()

total_stations = len(all_stations)
print(f"Total unique stations: {total_stations}")

all_stations.to_parquet("Geo/all_stations.parquet", index=False)

all_stations

Total unique stations: 233516


,decimalLatitudeStart,decimalLongitudeStart,datasetKey,datasetTitle,sourceDatabase,year,gebcoDepth,startDayOfYear,gear
0,70.001883,-126.254916,83eac1c3-e549-4eba-9dad-01d81ba8b94c,DFO Arctic Region Biodiversity of the Benthic ...,OBIS,2022,-212.0,230,<NA>
1,63.718060,-68.518610,8dbd0669-92d1-41ef-a5db-527bc6aa1ec9,Biodiversity of benthic assemblages on the Arc...,OBIS,1969,4.0,246,<NA>
2,53.292600,-169.778670,f5a4799e-dc24-4807-89d9-01da47d52e3b,"NOAA, Deep-Sea Coral Research and Technology P...",OBIS,2023,-792.0,204,<NA>
3,55.740002,-162.823334,89e23fc8-3f61-4480-9de3-358fe6eefe0b,National Museum of Natural History Invertebrat...,OBIS,1982,-57.0,304,<NA>
4,72.351167,-119.073333,55ff4bf2-b3d2-49bf-8749-7c41fb2c0c19,DFO Arctic Region Biodiversity of the Benthic ...,OBIS,2024,-77.0,238,<NA>
...,...,...,...,...,...,...,...,...,...
3734287,80.450000,-128.980000,Benthos ArcOD Sirenko (a),Occurrences of macro- and megabenthic organism...,CRITTERBASE,1977,-3510.0,193,Sigsbi trawl generic
3734288,80.130000,-127.800000,Benthos ArcOD Sirenko (a),Occurrences of macro- and megabenthic organism...,CRITTERBASE,1977,-2279.0,199,Sigsbi trawl generic
3734289,79.430000,-126.370000,Benthos ArcOD Sirenko (a),Occurrences of macro- and megabenthic organism...,CRITTERBASE,1977,-2515.0,221,Sigsbi trawl generic
3734290,79.370000,-127.700000,Benthos ArcOD Sirenko (a),Occurrences of macro- and megabenthic organism...,CRITTERBASE,1977,-3395.0,240,grab generic


In [3]:
# helper: drop rings < 4 points and invalid geometries  
def clean_poly(g):  
    if g is None: return None  
    try: g = make_valid(g)  
    except Exception: return None  
    if g.is_empty or not g.is_valid: return None  
    if g.geom_type == "Polygon":  
        return g if len(g.exterior.coords) >= 4 else None  
    if g.geom_type == "MultiPolygon":  
        parts = [p for p in g.geoms if len(p.exterior.coords) >= 4]  
        if not parts: return None  
        return MultiPolygon(parts) if len(parts) > 1 else parts[0]  
    return None  
  
# 1 ▸ device points  
df  = pd.read_parquet("Geo/all_stations.parquet")  
pts = gpd.GeoDataFrame(  
    df,  
    geometry=[Point(xy) for xy in zip(df["decimalLongitudeStart"], df["decimalLatitudeStart"])],  
    crs="EPSG:4326",  
)  
print("Station rows:", len(pts))  
  
# 2 ▸ shapefiles in Geo/  
shp_dir = pathlib.Path("Geo")  
shps    = sorted(shp_dir.glob("*.shp"))  
print("Found", len(shps), "shapefiles in Geo/")  
  
joined = pts.copy()  
  
for shp in shps:  
    name = shp.stem  
    print("•", name)  
    raw = gpd.read_file(shp)  
    prop_cols = [c for c in raw.columns if c.lower() in ["class", "geomorphic"]]  
  
    # clean layer  
    records = []  
    for _, row in tqdm(raw.iterrows(), total=len(raw), leave=False):  
        geom = clean_poly(row.geometry)  
        if geom is None: continue  
        rec  = {name + "_" + c: row[c] for c in prop_cols}  
        rec["geometry"] = geom  
        records.append(rec)  
  
    if not records:  
        print("  ↳ skipped (no valid polygons)"); continue  
  
    layer = gpd.GeoDataFrame(records, crs=raw.crs).to_crs("EPSG:4326")  
    joined = gpd.sjoin(joined, layer, how="left", predicate="intersects").drop(columns="index_right")  

# Handle duplicate columns by renaming with suffix - after all joins complete
print("Checking for duplicate columns...")
cols = joined.columns.tolist()
new_cols = []
seen = {}
for col in cols:
    if col in seen:
        seen[col] += 1
        new_cols.append(f"{col}_{seen[col]}")
        print(f"  Renamed duplicate '{col}' → '{col}_{seen[col]}'")
    else:
        seen[col] = 0
        new_cols.append(col)
joined.columns = new_cols
  
# 3 ▸ reorder columns  
order = [  
    "Hadal", "Abyss", "Slope", "Shelf", "Abyssal_Classification", "Shelf_Classification",  
    "Plateaus", "Terraces", "Trenches", "Rises", "Fans", "Spreading_ridges", "Ridges",  
    "Troughs", "Rift_valleys", "Shelf_valleys", "Glacial_troughs", "Sills",  
    "Bridges", "Seamounts", "Guyots", "Canyons", "Escarpments", "Basins"  
]  
  
# build full ordered list: point columns first, then *_Class/_Geomorphic in order  
base_cols   = [c for c in joined.columns if c not in joined.geometry.name and not any(c.startswith(x + "_") for x in order)]  
ordered_cols = base_cols.copy()  
added = set(base_cols)  # Track which columns we've already added
for feat in order:  
    for c in joined.columns:
        if c.startswith(feat + "_") and c not in added:
            ordered_cols.append(c)
            added.add(c)
  
joined = joined[ordered_cols + [joined.geometry.name]]  
  
# 4 ▸ save + preview  
out_parquet = "Geo/all_stations.parquet"  
joined.drop(columns="geometry").to_parquet(out_parquet, index=False)  
print("✓ saved to", out_parquet)  
print(joined.drop(columns='geometry').head())

Station rows: 233516
Found 24 shapefiles in Geo/
• Abyss


• Abyssal_Classification


• Basins


• Bridges


• Canyons


• Escarpments


... (16 outputs left out)

• Trenches


• Troughs


Checking for duplicate columns...


✓ saved to Geo/all_stations.parquet
   decimalLatitudeStart  decimalLongitudeStart  \
0             70.001883            -126.254916   
1             63.718060             -68.518610   
2             53.292600            -169.778670   
3             55.740002            -162.823334   
4             72.351167            -119.073333   

                             datasetKey  \
0  83eac1c3-e549-4eba-9dad-01d81ba8b94c   
1  8dbd0669-92d1-41ef-a5db-527bc6aa1ec9   
2  f5a4799e-dc24-4807-89d9-01da47d52e3b   
3  89e23fc8-3f61-4480-9de3-358fe6eefe0b   
4  55ff4bf2-b3d2-49bf-8749-7c41fb2c0c19   

                                        datasetTitle sourceDatabase  year  \
0  DFO Arctic Region Biodiversity of the Benthic ...           OBIS  2022   
1  Biodiversity of benthic assemblages on the Arc...           OBIS  1969   
2  NOAA, Deep-Sea Coral Research and Technology P...           OBIS  2023   
3  National Museum of Natural History Invertebrat...           OBIS  1982   
4  DFO Arctic Regio

In [4]:
input_file = "Geo/all_stations.parquet"   
df_geo = pd.read_parquet(input_file)  
  
print("Rows:", len(df_geo))  

# 2. Create the target column and initialize with empty strings  
df_geo["geomorphologicalCategory"] = ""  

# 3.1  Shelf + high  ➜  Deep shelf  
mask = (  
    df_geo["Shelf_Geomorphic"].str.contains("Shelf", case=False, na=False)  
    & df_geo["Shelf_Classification_Class"].str.contains("high", case=False, na=False)  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Deep shelf"  
  
# 3.2  Shelf + medium  ➜  Medium shelf  
mask = (  
    df_geo["Shelf_Geomorphic"].str.contains("Shelf", case=False, na=False)  
    & df_geo["Shelf_Classification_Class"].str.contains("medium", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Medium shelf"  
  
# 3.3  Shelf + low  ➜  Shallow shelf  
mask = (  
    df_geo["Shelf_Geomorphic"].str.contains("Shelf", case=False, na=False)  
    & df_geo["Shelf_Classification_Class"].str.contains("low", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Shallow shelf"  
  
# 3.4  Slope  ➜  Slope  
mask = (  
    df_geo["Slope_Geomorphic"].str.contains("Slope", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Slope"  
  
# 3.5  Ridges_Geomorphic with “Ridge”  ➜  Ridges  
mask = (  
    df_geo["Ridges_Geomorphic"].str.contains("Ridge", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Ridges"  
  
# 3.6  Spreading_ridges_Geomorphic with “Spreading Ridge”  ➜  Ridges  
mask = (  
    df_geo["Spreading_ridges_Geomorphic"].str.contains("Spreading Ridge", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Ridges"  
  
# 3.7  Rift_valleys_Geomorphic with “Rift Valley”  ➜  Ridges  
mask = (  
    df_geo["Rift_valleys_Geomorphic"].str.contains("Rift Valley", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Ridges"  
  
# 3.8  Abyssal_Classification_Class with “Hills” or “Mountains”  ➜  Abyssal mountains and hills
mask = (  
    df_geo["Abyssal_Classification_Class"].str.contains("Hills|Mountains", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Abyssal mountains and hills"  
  
# 3.9  Seamounts_Geomorphic with “Seamount”  ➜  Abyssal mountains and hills 
mask = (  
    df_geo["Seamounts_Geomorphic"].str.contains("Seamount", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Abyssal mountains and hills"  
  
# 3.10  Abyssal_Classification_Class with “Plains”  ➜  Abyssal plains and hadal zones
mask = (  
    df_geo["Abyssal_Classification_Class"].str.contains("Plains", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Abyssal plains and hadal zones"  
  
# 3.11  Hadal_Geomorphic with “Hadal”  ➜  Abyssal plains and hadal zones  
mask = (  
    df_geo["Hadal_Geomorphic"].str.contains("Hadal", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Abyssal plains and hadal zones"  

# Save as excel
output_file = "Geo/all_stations.parquet"  
df_geo.to_parquet(output_file, index=False)  
  
print("Saved new file:", output_file)  

Rows: 233516
Saved new file: Geo/all_stations.parquet


In [5]:
# Delete unneccessary columns
# Load existing collapsed file 
input_file = 'Geo/all_stations.parquet'
df_geo = pd.read_parquet(input_file)

# Retain only desired columns (remove sourceDatabase to avoid duplicates)
keep_cols = ['decimalLatitudeStart', 'decimalLongitudeStart', 'geomorphologicalCategory']
df_geo = df_geo[keep_cols]

# Deduplicate on (lat, lon, year) - same location should have one geomorphologicalCategory
df_geo = df_geo.drop_duplicates(subset=['decimalLatitudeStart', 'decimalLongitudeStart'], keep='first')

# Save to a new file
output_file = 'Geo/all_stations.parquet'
df_geo.to_parquet(output_file, index=False)

# Show head for confirmation
print(df_geo.head())
print('Saved file as', output_file)
print(f'Total unique station-years: {len(df_geo)}')

   decimalLatitudeStart  decimalLongitudeStart geomorphologicalCategory
0             70.001883            -126.254916             Medium shelf
1             63.718060             -68.518610               Deep shelf
2             53.292600            -169.778670                    Slope
3             55.740002            -162.823334             Medium shelf
4             72.351167            -119.073333            Shallow shelf
Saved file as Geo/all_stations.parquet
Total unique station-years: 183489


In [6]:
# Calculate nr of rows that are lacking a geo category

df = pd.read_parquet('Geo/all_stations.parquet')

missing_mask = df['geomorphologicalCategory'].isna() | (df['geomorphologicalCategory'].str.strip() == '')
missing_count = missing_mask.sum()
print('Missing rows count:', missing_count)

Missing rows count: 13458


In [7]:
# Load the two parquet files
all_merged = pd.read_parquet("GEBCO/merged_all_gebco.parquet")
geo_collapsed = pd.read_parquet("Geo/all_stations.parquet")

# Merge on the combination of decimalLatitudeStart, decimalLongitudeStart
all_merged_geo = all_merged.merge(
    geo_collapsed[['decimalLatitudeStart', 'decimalLongitudeStart', 'geomorphologicalCategory']],
    on=['decimalLatitudeStart', 'decimalLongitudeStart'],
    how='left'
)

# Save the merged dataframe
all_merged_geo.to_parquet("Geo/merged_all_geo.parquet", index=False)

print(f"Merged dataframe shape: {all_merged_geo.shape}")
print(f"Original all_merged: {len(all_merged)} records")
print(f"geomorphologicalCategory column added with {all_merged_geo['geomorphologicalCategory'].notna().sum()} non-null values")
print(all_merged_geo[['decimalLatitudeStart', 'decimalLongitudeStart', 'geomorphologicalCategory']].head(10))

Merged dataframe shape: (3736407, 68)
Original all_merged: 3736407 records
geomorphologicalCategory column added with 3736407 non-null values
   decimalLatitudeStart  decimalLongitudeStart geomorphologicalCategory
0             70.001883            -126.254916             Medium shelf
1             63.718060             -68.518610               Deep shelf
2             53.292600            -169.778670                    Slope
3             55.740002            -162.823334             Medium shelf
4             72.351167            -119.073333            Shallow shelf
5             63.097661              -8.587832                    Slope
6             56.400000            -165.586700            Shallow shelf
7             67.830002            -165.419998            Shallow shelf
8             69.570000            -134.050003             Medium shelf
9             51.681068            -179.584763                    Slope


In [8]:
# Count records per geomorphologicalCategory in all_merged_geo.parquet
all_merged_geo = pd.read_parquet("Geo/merged_all_geo.parquet")

geo_category_counts = all_merged_geo['geomorphologicalCategory'].value_counts(dropna=False)

print("Records per geomorphologicalCategory:")
print(geo_category_counts)
print(f"\nTotal records: {len(all_merged_geo)}")

Records per geomorphologicalCategory:
geomorphologicalCategory
Medium shelf                      1614034
Deep shelf                        1286883
Shallow shelf                      303319
Slope                              274279
                                   171475
Abyssal mountains and hills         45439
Ridges                              24879
Abyssal plains and hadal zones      16099
Name: count, dtype: int64

Total records: 3736407
